In [1]:
"""
Примеры кода к Лекции 5.1: От цепочек к графам — почему линейности недостаточно

Этот модуль демонстрирует:
1. Минимальный граф состояний (StateGraph) с двумя узлами
2. Условные рёбра и циклы (генерация шуток с повторами)
3. Граф как Runnable — интеграция с LCEL-цепочками
"""

'\nПримеры кода к Лекции 5.1: От цепочек к графам — почему линейности недостаточно\n\nЭтот модуль демонстрирует:\n1. Минимальный граф состояний (StateGraph) с двумя узлами\n2. Условные рёбра и циклы (генерация шуток с повторами)\n3. Граф как Runnable — интеграция с LCEL-цепочками\n'

In [2]:
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import END, START, MessagesState, StateGraph
from llm_config import check_api_key, get_llm

In [3]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Минимальный граф — привет, мир

In [4]:
class JokeState(TypedDict):
    """Состояние для простого графа генерации и оценки шуток."""

    topic: str
    joke: str
    score: int


# Узел генерации — принимает состояние, возвращает обновления
def generate_joke(state: JokeState) -> dict:
    topic = state["topic"]
    return {
        "joke": f"Почему {topic} перешёл дорогу? Чтобы попасть на другую сторону!"
    }

# Узел оценки — упрощённая «оценка» шутки
def evaluate_joke(state: JokeState) -> dict:
    joke = state["joke"]
    score = 3 if len(joke) > 50 else 1
    return {"score": score}

# Сборка графа: узлы + рёбра + компиляция
graph = StateGraph(JokeState)

graph.add_node("generate", generate_joke)
graph.add_node("evaluate", evaluate_joke)

graph.add_edge(START, "generate")  # Старт → генерация
graph.add_edge("generate", "evaluate")  # Генерация → оценка
graph.add_edge("evaluate", END)  # Оценка → конец

app = graph.compile()

# Запуск графа
result = app.invoke({"topic": "программист"})
print(f"\nРезультат: {result}")
print(f"  Тема: {result['topic']}")
print(f"  Шутка: {result['joke']}")
print(f"  Оценка: {result['score']}")


Результат: {'topic': 'программист', 'joke': 'Почему программист перешёл дорогу? Чтобы попасть на другую сторону!', 'score': 3}
  Тема: программист
  Шутка: Почему программист перешёл дорогу? Чтобы попасть на другую сторону!
  Оценка: 3


## Условные рёбра и циклы

In [5]:
class JokeLoopState(TypedDict):
    """Состояние для графа с циклом: генерация → оценка → повтор."""

    topic: str
    joke: str
    feedback: str
    attempts: int


llm = get_llm()

def generate_joke(state: JokeLoopState) -> dict:
    attempts = state.get("attempts", 0)
    feedback = state.get("feedback", "")

    messages = [
        SystemMessage(content="Ты — комик. Придумай короткую смешную шутку."),
        HumanMessage(content=f"Тема: {state['topic']}"),
    ]

    if feedback:
        messages.append(
            HumanMessage(
                content=f"Предыдущая шутка была плохой. Критика: {feedback}. Придумай другую."
            )
        )

    response = llm.invoke(messages)
    return {"joke": response.content, "attempts": attempts + 1}

def evaluate_joke(state: JokeLoopState) -> dict:
    response = llm.invoke(
        [
            SystemMessage(
                content="Оцени шутку. Ответь ОДНИМ словом: FUNNY или BORING."
            ),
            HumanMessage(content=state["joke"]),
        ]
    )
    verdict = response.content.strip().upper()
    feedback = "" if "FUNNY" in verdict else "Шутка скучная, нужно смешнее"
    return {"feedback": feedback}

# Функция-маршрутизатор: решает, куда идти после оценки
def should_continue(state: JokeLoopState) -> str:
    if not state.get("feedback"):  # Пустая строка = шутка хорошая
        return "end"
    if state.get("attempts", 0) >= 3:  # Лимит попыток
        return "end"
    return "retry"

# Сборка графа
graph = StateGraph(JokeLoopState)
graph.add_node("generate", generate_joke)
graph.add_node("evaluate", evaluate_joke)
graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
# Условное ребро: после evaluate — либо END, либо обратно в generate
graph.add_conditional_edges(
    "evaluate",
    should_continue,
    {
        "end": END,
        "retry": "generate",
    },
)
app = graph.compile()

result = app.invoke({"topic": "искусственный интеллект"})
print(f"\nШутка (за {result['attempts']} попыток): {result['joke']}")


Шутка (за 1 попыток): Искусственный интеллект — это когда программа уже умеет писать стихи, а человек всё ещё пытается понять, куда делся зарядник.


## Граф как Runnable — интеграция с LCEL

In [6]:
llm = get_llm()

def chatbot(state: MessagesState) -> dict:
    system = SystemMessage(content="Ты — дружелюбный ассистент. Отвечай кратко.")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}

graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()

# Граф внутри LCEL-цепочки
chain = app | (lambda state: state["messages"][-1].content) | StrOutputParser()

print("\n3.1. Граф встроен в LCEL-цепочку:")
result = chain.invoke({"messages": [("user", "Привет!")]})
print(f"  Результат (строка): {result}")


3.1. Граф встроен в LCEL-цепочку:


  Результат (строка): Привет! Чем могу помочь?
